# Lekcija 04 - Dizajnerski obrazac korištenja alata

U ovoj lekciji naučit ćete dizajnerski obrazac **Korištenje alata** za AI agente koristeći Microsoft Agent Framework (Python). Obrađujemo:

- Definiranje funkcijskih alata s dekoratorom `@tool` i tipiziranim parametrima
- Davanje shema alata kako bi model znao što svaki alat radi
- Kontrolu izvršavanja alata s `approval_mode`
- Vraćanje **strukturiranog izlaza** putem Pydantic modela i `response_format`

Scenarij je **agent za rezervaciju putovanja** koji može pregledavati destinacije, provjeravati dostupnost i dohvaćati informacije o letovima.


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Definiranje alata s dekoratorom @tool

Dekorator `@tool` pretvara običnu Python funkciju u alat koji agent može pozvati.  
Ključne točke:

- **Docstring** postaje opis alata koji model vidi.  
- **Tipovi anotacija** (uključujući `Annotated` s opisima) definiraju shemu alata.  
- `approval_mode` kontrolira mora li korisnik odobriti svaki poziv prije nego što se izvrši.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Izrada agenta s više alata

Proslijedite sva tri alata klijentu kako bi model mogao pozvati one koje treba za odgovor na pitanje korisnika.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturirani izlaz s alatima

Postavljanjem `response_format` na Pydantic model, agent je primoran vratiti dobro tipizirani JSON objekt umjesto slobodnog teksta. Ovo je korisno kada kod koji slijedi treba programski konzumirati rezultat.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Obrasci odobravanja alata

Parametar `approval_mode` na `@tool` kontrolira zahtijevaju li pozivi alata ljudsko odobrenje prije izvršavanja:

| Način | Ponašanje |
|---|---|
| `"never_require"` | Alat se izvršava automatski — nije potrebna potvrda korisnika. |
| `"always_require"` | Svaki poziv mora biti odobren od strane korisnika prije izvršenja. |

Koristite `"always_require"` za alate koji imaju nuspojave (npr. rezervacija leta, terećenje kreditne kartice) kako bi čovjek ostao uključen u proces.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Sažetak

U ovoj lekciji naučili ste kako:

1. **Definirati alate** koristeći dekorator `@tool` s tipiziranim parametrima i docstringovima koji služe kao shema alata.
2. **Sastaviti više alata** tako da agent može pozivati ih jedan za drugim kako bi odgovorio na složene upite.
3. **Vratiti strukturirani izlaz** prosljeđivanjem Pydantic modela kao `response_format`.
4. **Kontrolirati odobrenje alata** pomoću `approval_mode` kako bi se zadržala ljudska kontrola u osjetljivim operacijama.

Ovi obrasci čine temelj za izgradnju pouzdanih agenata spremnih za proizvodnju koji mogu sigurno komunicirati s vanjskim sustavima.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
